# v1 — Baseline: frequency-encoding + top-15 mutual information + XGBoost

**Hypothesis.** A plain pipeline (median impute → scale → frequency-encode categoricals → keep the top-15 features by mutual information → XGBoost) already gets most of the available signal, given how strong features like `quantity` are (see `00_data`).

**What this notebook adds beyond the original v1 script.**
1. The script scored a *single 70/30 holdout*. Here the same pipeline runs on the shared 5-fold CV.
2. Its feature step (imputer, scaler, **mutual-information selection using the labels**) was fit on *all* rows before the holdout split. That is a leak. We measure how large it is.
3. We test whether the top-15 MI cut helps or hurts.

| row | what changes |
|---|---|
| A0 | v1 *as shipped*: features fit on all rows, then CV |
| A1 | **leak-free**: features (incl. MI selection) fit inside each training fold |
| A2 | A1 without the top-15 cut: keep all features |

In [1]:
import sys, warnings
sys.path.insert(0, '.')
warnings.simplefilter('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import *
pd.set_option('display.width', 140)
V1 = load_version('v1')
df, y, le = get_train('v1')
P1 = dict(n_estimators=300, max_depth=8, learning_rate=0.1)     # v1's XGBoost params
print(df.shape, le.classes_)

(59400, 29) ['functional' 'functional needs repair' 'non functional']


## A0 — v1 as shipped (features fit on all rows, label-aware selection included)

In [2]:
from sklearn.metrics import accuracy_score
import xgboost as xgb
X_all = V1.FeaturePipeline().fit_transform(df.drop(columns=[TARGET]), y)     # sees every row and label
print('selected features:', list(X_all.columns))
sc = []
for tr, va in folds(y):
    clf = xgb.XGBClassifier(**P1, tree_method='hist', random_state=SEED, n_jobs=-1).fit(X_all.iloc[tr], y[tr])
    sc.append(accuracy_score(y[va], clf.predict(X_all.iloc[va])))
r = log_result('v1', 'A0 as shipped (leaky feature fit)', sc); print(f"{r['mean']:.4f} ± {r['std']:.4f}")

selected features: ['quantity', 'lga', 'waterpoint_type', 'longitude', 'funder', 'latitude', 'extraction_type_class', 'installer', 'region_code', 'ward', 'region', 'amount_tsh', 'scheme_name', 'construction_year', 'payment']


0.8037 ± 0.0045


## A1 — leak-free: feature pipeline fit inside each fold

In [3]:
sc, oof_a1 = cv_xgb(df, y, V1.FeaturePipeline, P1, y_to_features=True)
r = log_result('v1', 'A1 leak-free', sc); print(f"{r['mean']:.4f} ± {r['std']:.4f}")

0.8034 ± 0.0044


## A2 — drop the top-15 mutual-information cut

In [4]:
class NoCut(V1.FeaturePipeline):
    def _select_features(self, df, fit, y):
        return df[self.continuous_cols_ + self.categorical_cols_]

sc, _ = cv_xgb(df, y, NoCut, P1, y_to_features=True)
r = log_result('v1', 'A2 no MI cut (keep all)', sc); print(f"{r['mean']:.4f} ± {r['std']:.4f}")

0.8090 ± 0.0034


## Results

In [5]:
res = pd.DataFrame([r for r in map(__import__('json').loads, RESULTS.read_text().splitlines()) if r['version'] == 'v1'])
res['acc'] = res['mean'].round(4).astype(str) + ' ± ' + res['std'].round(4).astype(str)
res[['label', 'acc']].set_index('label')

,acc
label,
A0 as shipped (leaky feature fit),0.8037 ± 0.0045
A1 leak-free,0.8034 ± 0.0044
A2 no MI cut (keep all),0.809 ± 0.0034


In [6]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y, oof_a1))
print(classification_report(y, oof_a1, target_names=le.classes_, digits=3))

[[28947   609  2703]
 [ 2309  1326   682]
 [ 5049   324 17451]]


                         precision    recall  f1-score   support

             functional      0.797     0.897     0.844     32259
functional needs repair      0.587     0.307     0.403      4317
         non functional      0.838     0.765     0.799     22824

               accuracy                          0.803     59400
              macro avg      0.741     0.656     0.682     59400
           weighted avg      0.797     0.803     0.795     59400



## Conclusion

| row | CV accuracy (mean ± std) | vs A1 |
|---|---|---|
| A0 as shipped (leaky feature fit) | 0.8037 ± 0.0045 | +0.0003 |
| A1 leak-free | 0.8034 ± 0.0044 | — |
| A2 no MI cut | **0.8090 ± 0.0034** | **+0.0056** |

- **The feature-fit leak is negligible.** A0 vs A1 differs by +0.0003, far below the fold std (0.0044), and the per-fold sign flips (+0.0005, −0.0031, −0.0007, −0.0008, +0.0056). The MI selection runs over only ~29 columns, so it can't overfit the labels much. The original v1 script's protocol was sloppy but harmless here. (I'd not assume that for a pipeline with many more features.)
- **The top-15 MI cut hurts.** Dropping it gains +0.0056, and **all 5 folds improve** (+0.004 to +0.007). That's about 1.3× the fold std, but the paired, same-fold, same-direction gain is more convincing than the std comparison. Likely cause: the MI score is computed on frequency-encoded columns, which is a poor measure for tree models, and it throws away columns that XGBoost could use. Lesson: don't pre-select features for a tree ensemble.
- **Where v1 fails:** *functional needs repair* has recall **0.307** (F1 0.40) versus 0.90 and 0.77 for the other classes. Most missed repair pumps go to *functional* (2,309 of 4,317). Accuracy hides this because the class is only 7% of rows.

**Baseline to beat: ≈ 0.803 (A1), or 0.809 with all features.**

Next: **v2** replaces frequency encoding with native categoricals and adds geo-imputation, missingness flags and class weights, one change at a time.